# Set Stats vs YTD Stats Analysis

Two questions:
1. **How different are per-set stats from YTD stats?** — tells us how well YTD anchors a single-set performance
2. **How much do stats vary across sets within the same match?** — tells us if momentum/fatigue effects are real in the data

Only 2026 data (set_stats only exist for 2026).

In [1]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = 'tennis.db'

STAT_COLS = [
    'first_serve_pct',
    'first_serve_won_pct',
    'second_serve_won_pct',
    'bp_saved_pct',
    'bp_converted_pct',
    'first_return_won_pct',
    'second_return_won_pct',
    'total_svc_pts_won_pct',
    'total_pts_won_pct',
]

conn = sqlite3.connect(DB_PATH)

# Join set_stats with match_ytd_stats on (match_id, player_id)
set_cols_sql  = ', '.join(f's.{c} AS set_{c}' for c in STAT_COLS)
ytd_cols_sql  = ', '.join(f'y.{c} AS ytd_{c}' for c in STAT_COLS)

query = f"""
SELECT
    s.match_id, s.player_id, s.set_number,
    {set_cols_sql},
    {ytd_cols_sql}
FROM set_stats s
JOIN match_ytd_stats y ON y.match_id = s.match_id AND y.player_id = s.player_id
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f'Rows joined: {len(df):,}  (expect ~5,788)')
print(f'Set numbers present: {sorted(df["set_number"].unique())}')

Rows joined: 5,788  (expect ~5,788)
Set numbers present: [np.int64(1), np.int64(2), np.int64(3)]


## Analysis 1: How different are set stats from YTD stats?

For each stat and set number, compute the mean absolute difference (set value − YTD value).
Values are 0–100 percentage points.

In [2]:
# Compute abs diff and signed diff for each stat
for c in STAT_COLS:
    df[f'diff_{c}']     = df[f'set_{c}'] - df[f'ytd_{c}']
    df[f'absdiff_{c}']  = df[f'diff_{c}'].abs()

absdiff_cols = [f'absdiff_{c}' for c in STAT_COLS]
diff_cols    = [f'diff_{c}'    for c in STAT_COLS]

# Mean absolute difference by set_number
mad_by_set = df.groupby('set_number')[absdiff_cols].mean().round(1)
mad_by_set.columns = STAT_COLS
mad_by_set.index.name = 'set'

print('Mean Absolute Difference (set stat − YTD stat), by set number')
print('Values in percentage points (0–100 scale)')
print()
print(mad_by_set.T.to_string())
print()
print('Overall mean absolute diff (all sets):')
print(df[absdiff_cols].mean().round(1).rename(lambda c: c.replace('absdiff_','')))

Mean Absolute Difference (set stat − YTD stat), by set number
Values in percentage points (0–100 scale)

set                       1     2     3
first_serve_pct         8.2   8.0   8.7
first_serve_won_pct    10.7  11.0  10.7
second_serve_won_pct   14.4  14.3  14.7
bp_saved_pct           36.7  36.2  37.6
bp_converted_pct       31.4  30.4  31.5
first_return_won_pct   10.5  10.9  10.4
second_return_won_pct  14.2  14.5  14.3
total_svc_pts_won_pct   9.6   9.8   9.9
total_pts_won_pct       7.2   7.5   7.3

Overall mean absolute diff (all sets):
first_serve_pct           8.2
first_serve_won_pct      10.8
second_serve_won_pct     14.4
bp_saved_pct             36.6
bp_converted_pct         31.0
first_return_won_pct     10.6
second_return_won_pct    14.4
total_svc_pts_won_pct     9.7
total_pts_won_pct         7.3
dtype: float64


In [3]:
# Std of differences — how spread out is the disagreement?
std_by_set = df.groupby('set_number')[diff_cols].std().round(1)
std_by_set.columns = STAT_COLS
std_by_set.index.name = 'set'

print('Std of (set stat − YTD stat), by set number')
print()
print(std_by_set.T.to_string())

Std of (set stat − YTD stat), by set number

set                       1     2     3
first_serve_pct        11.5  11.6  12.4
first_serve_won_pct    14.5  15.2  14.8
second_serve_won_pct   18.5  18.4  19.0
bp_saved_pct           40.0  39.8  39.3
bp_converted_pct       37.4  36.4  37.5
first_return_won_pct   13.5  14.0  13.2
second_return_won_pct  18.3  18.8  18.7
total_svc_pts_won_pct  13.0  13.5  13.5
total_pts_won_pct       9.8  10.3  10.1


## Analysis 2: Within-match variation across sets

For each (match, player) pair with ≥2 sets, compute the **std of each stat across set numbers**.
A high std means the player performed very differently from set to set.

In [4]:
set_only_cols = [f'set_{c}' for c in STAT_COLS]

# Std across sets for each (match_id, player_id)
within_std = (
    df.groupby(['match_id', 'player_id'])[set_only_cols]
    .std()
    .dropna(how='all')   # drop pairs with only 1 set (std=NaN)
)
within_std.columns = STAT_COLS

n_pairs = len(within_std)
print(f'(match, player) pairs with ≥2 sets: {n_pairs:,}')
print()

summary = within_std.agg(['mean','median','max']).round(1)
print('Within-match std across sets (percentage points):')
print(summary.to_string())

(match, player) pairs with ≥2 sets: 2,462

Within-match std across sets (percentage points):
        first_serve_pct  first_serve_won_pct  second_serve_won_pct  bp_saved_pct  bp_converted_pct  first_return_won_pct  second_return_won_pct  total_svc_pts_won_pct  total_pts_won_pct
mean                7.6                  9.3                  14.1          30.9              25.9                   9.2                   14.0                    8.1                5.5
median              6.5                  7.8                  12.7          29.7              23.3                   7.8                   12.7                    7.1                4.7
max                60.1                 70.7                  70.7          70.7              70.7                  57.3                   70.7                   65.1               53.0


In [5]:
# Distribution: what fraction of (match, player) pairs have std > 10 pts on key stats?
print('Fraction of (match, player) pairs with within-match std > 10 ppts:')
for c in STAT_COLS:
    frac = (within_std[c].dropna() > 10).mean()
    print(f'  {c:<28} {frac*100:.1f}%')

Fraction of (match, player) pairs with within-match std > 10 ppts:
  first_serve_pct              26.8%
  first_serve_won_pct          37.9%
  second_serve_won_pct         60.0%
  bp_saved_pct                 76.5%
  bp_converted_pct             71.3%
  first_return_won_pct         37.7%
  second_return_won_pct        60.0%
  total_svc_pts_won_pct        31.1%
  total_pts_won_pct            14.1%


In [6]:
# Set 1 vs Set 2 vs Set 3 average values — does performance drift over the match?
set_means = df.groupby('set_number')[[f'set_{c}' for c in STAT_COLS]].mean().round(1)
set_means.columns = STAT_COLS
set_means.index.name = 'set'

print('Average stat value by set number (does performance drift?)')
print(set_means.T.to_string())

Average stat value by set number (does performance drift?)
set                       1     2     3
first_serve_pct        63.4  63.5  62.7
first_serve_won_pct    72.0  71.4  71.7
second_serve_won_pct   51.5  51.4  51.7
bp_saved_pct           43.5  43.7  40.3
bp_converted_pct       35.1  35.2  35.4
first_return_won_pct   28.0  28.4  27.7
second_return_won_pct  48.6  48.3  47.7
total_svc_pts_won_pct  64.6  64.1  64.2
total_pts_won_pct      50.0  49.9  49.7
